# The Limits of Context Stuffing & Why Retrieval (RAG) Fixes them

## 4.1 The Real Problem: "We Have Text. Now What?"

After extraction, teams often experience a brief sense of victory.

> "The document is in Markdown. We're done."

But text alone does not create intelligence. A common first instinct is the **"just add it to context"** approach: take the entire document, paste it into the prompt, ask the model a question.

We already proved in the previous notebook that this fails — the document exceeds the deployed context window (120k tokens). Even when we force it to fit by truncating, we're paying a massive token cost for a simple question.

**The model doesn't need the whole book. It needs the right paragraph.**

That's retrieval.

### Load the environment variables

In [51]:
import sys
sys.path.insert(0, "..")
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base

url_models = f"{endpoint_base}/models"
url_chat = f"{endpoint_base}/chat/completions"

### 4.1 Understanding Tokens: The Unit That Actually Matters

#### Install tiktoken

In [52]:
! pip install tiktoken -q

In [53]:
import tiktoken

# Load your markdown file
with open("./Basic-Fantasy-RPG-Rules-r142.md", "r", encoding="utf-8") as f:
    text = f.read()

# Choose model encoding
encoding = tiktoken.get_encoding("cl100k_base")

# If you're encoding for gpt-4o
# encoding = tiktoken.encoding_for_model("gpt-4o")  # or "gpt-4"

tokens = encoding.encode(text)

print(f"Number of tokens: {len(tokens)}")

Number of tokens: 205443


## Model List

granite-3-2-8b-instruct has 4 million token context length, enough for our puposes here.

In [54]:
import requests

# Load markdown file
with open("Basic-Fantasy-RPG-Rules-r142.md", "r", encoding="utf-8") as f:
    document_text = f.read()

headers = {
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json"
}

data = {
    "model": "granite-3-2-8b-instruct",
    "messages": [
        {
            "role": "system",
            "content": "Answer the question using only the provided document."
        },
        {
            "role": "user",
            "content": f"""
You are given the following document:

{document_text}

Question:
What happens if a Thief fails an Open Locks attempt?
"""
        }
    ]
}

response = requests.post(url_chat, headers=headers, json=data)

print(response.json())


{'error': {'message': "litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 120000 tokens. However, you requested 241974 tokens in the messages, Please reduce the length of the messages. None\nmodel=granite-3-2-8b-instruct. context_window_fallbacks=None. fallbacks=None.\n\nSet 'context_window_fallback' - https://docs.litellm.ai/docs/routing#fallbacks. Received Model Group=granite-3-2-8b-instruct\nAvailable Model Group Fallbacks=None", 'type': None, 'param': None, 'code': '400'}}


## Let's Measure Tokens

In [55]:
import requests
import tiktoken
import json

source_md = "Basic-Fantasy-RPG-Rules-r142.md"

with open(source_md, "r", encoding="utf-8") as f:
    document_text = f.read()

question = "What happens if a Thief fails an Open Locks attempt?"

# Token estimate (good enough for a lab)
enc = tiktoken.get_encoding("cl100k_base")
doc_tokens = len(enc.encode(document_text))
question_tokens = len(enc.encode(question))

print(f"Document tokens (estimate): {doc_tokens:,}")
print(f"Question tokens (estimate):  {question_tokens:,}")
print(f"Total prompt tokens (estimate): {(doc_tokens + question_tokens):,}")
print("Expected max context (per error): 120,000 tokens\n")


Document tokens (estimate): 205,443
Question tokens (estimate):  12
Total prompt tokens (estimate): 205,455
Expected max context (per error): 120,000 tokens



## Let's reduce the number of tokens we're sending

In [56]:
MAX_CTX = 100_000
RESERVE = 2_000  # room for system + question + answer
BUDGET = MAX_CTX - RESERVE

tokens = enc.encode(document_text)
document_text_trimmed = enc.decode(tokens[:BUDGET])

print("Trimmed doc tokens:", len(enc.encode(document_text_trimmed)))

data["messages"][1]["content"] = f"""
You are given the following document:

{document_text_trimmed}

Question:
What happens if a Thief fails an Open Locks attempt?
"""
response = requests.post(url_chat, headers=headers, json=data)
print(response.status_code)
print(response.json())

Trimmed doc tokens: 98000
200
{'id': 'chatcmpl-21aea17e5de9424d8c6504aedcfe304b', 'created': 1771012658, 'model': 'granite-3-2-8b-instruct', 'object': 'chat.completion', 'system_fingerprint': None, 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': 'If a Thief fails an Open Locks attempt, the attempt does not automatically succeed on a subsequent try. They need to make the Open Locks check again each time they attempt to open the lock. A failed Open Locks attempt does not cause the lock to jam or become more difficult to open. It simply means that the current attempt has not succeeded in opening the lock.', 'role': 'assistant', 'tool_calls': None, 'function_call': None}, 'provider_specific_fields': {'stop_reason': None}}], 'usage': {'completion_tokens': 78, 'prompt_tokens': 114404, 'total_tokens': 114482, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'service_tier': None, 'prompt_logprobs': None, 'kv_transfer_params': None}


Just to be clear, here's the response

In [57]:
response.json()['choices'][0]['message']['content']

'If a Thief fails an Open Locks attempt, the attempt does not automatically succeed on a subsequent try. They need to make the Open Locks check again each time they attempt to open the lock. A failed Open Locks attempt does not cause the lock to jam or become more difficult to open. It simply means that the current attempt has not succeeded in opening the lock.'

And this time, the answer is the one we were looking for.

'If a Thief fails an Open Locks attempt, the Thief must wait until they have gained another level of experience before trying again.'

Yes, we can force it to fit. But we’re mutilating the input, paying a heavy token cost, and waiting longer than necessary for a simple answer.

This is not a failure of the model. It’s a failure of the design.

Remember, the model doesn’t need the whole book. It needs the right paragraph.
